# Excited States via Eigensolver and VQE

We compute excited state energies of H₂ using two approaches:
1. **Exact classical eigensolver** — diagonalize the full qubit Hamiltonian to get all eigenstates
2. **Eigensolver with VQE initial state** — use the VQE ground state as the starting point

Excited states are relevant for photoemission spectroscopy, charge transfer, and vibronic spectra.

## Step 1 — Imports

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True)

from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_algorithms.eigensolvers import NumPyEigensolver

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD
from qiskit_nature.second_q.circuit.library.initial_states import HartreeFock

from qiskit.quantum_info import Operator

print("All imports OK")

## Step 2 — H₂ Hamiltonian

In [ ]:
molecule = """
H 0.0 0.0 0.0
H 0.735 0.0 0.0
"""

driver = PySCFDriver(atom=molecule, basis="sto3g")
problem = driver.run()

num_spatial = problem.num_spatial_orbitals
num_particles = problem.num_particles
nuclear_repulsion = problem.nuclear_repulsion_energy()

hamiltonian = problem.hamiltonian
second_q_op = hamiltonian.second_q_op()
mapper = JordanWignerMapper()
qubit_op = mapper.map(second_q_op)

print(f"Spatial orbitals: {num_spatial}")
print(f"Qubits:           {qubit_op.num_qubits}")
print(f"Nuclear repulsion: {nuclear_repulsion:.6f} Ha")

## Step 3 — All Eigenvalues via Exact Diagonalization

In [ ]:
hamiltonian_matrix = qubit_op.to_matrix()
eigenvalues, eigenvectors = np.linalg.eigh(hamiltonian_matrix)

num_states = 8
all_energies = [
    eigenvalues[j] + nuclear_repulsion for j in range(num_states)
]

print("All Eigenvalues (Electronic + Nuclear Repulsion):")
print("-" * 45)
for j, E in enumerate(all_energies):
    label = "Ground State" if j == 0 else f"Excited State {j}"
    print(f"  {j}: {E:>14.10f} Ha  ({label})")

gs_energy = all_energies[0]
excited_energies = all_energies[1:]
print(f"\nGround state:       {gs_energy:.10f} Ha")
print(f"Excitation energies (relative to ground state):")
for j, E in enumerate(excited_energies):
    print(f"  S{j+1}: {E - gs_energy:>14.10f} Ha  ({E:.10f} Ha absolute)")

## Step 4 — Excited States via NumPyEigensolver

In [ ]:
eigensolver = NumPyEigensolver(k=6)
result = eigensolver.eigensolvers([qubit_op])

solver_energies = [ev + nuclear_repulsion for ev in result.eigenvalues]

print("NumPyEigensolver Results:")
for j, E in enumerate(solver_energies):
    label = "Ground State" if j == 0 else f"Excited {j}"
    print(f"  {j}: {E:>14.10f} Ha")

print(f"\nMatch with exact diagonalization: "
      f"{np.allclose(result.eigenvalues + nuclear_repulsion, all_energies[:6])}")

## Step 5 — VQE-Inspired Excited State via Subspace Expansion

Method: Given a VQE ground state |ψ₀⟩, we construct a subspace Hamiltonian
H_sub = ⟨ψ₀|O|ψₐ⟩ where |ψₐ⟩ = Oₐ|ψ₀⟩ are excited candidates generated by applying
Pauli excitation operators Oₐ to the ground state. Diagonalizing H_sub gives
approximate excited state energies without re-running VQE for each state.

We use the UCCSD excitation operators as our excitation basis.

In [ ]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import Estimator

initial_state = HartreeFock(
    num_spatial_orbitals=num_spatial,
    num_particles=num_particles,
    qubit_mapper=mapper,
)

ansatz = UCCSD(
    num_spatial_orbitals=num_spatial,
    num_particles=num_particles,
    qubit_mapper=mapper,
    initial_state=initial_state,
)

estimator = Estimator()
vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=COBYLA(maxiter=300))
result_vqe = vqe.compute_minimum_eigenvalue(qubit_op)

gs_energy_vqe = result_vqe.eigenvalue.real + nuclear_repulsion
print(f"VQE ground state energy: {gs_energy_vqe:.12f} Ha")
print(f"Exact ground state energy: {gs_energy:.12f} Ha")
print(f"Ground state error: {abs(gs_energy_vqe - gs_energy):.12f} Ha")

## Step 6 — Build Subspace Hamiltonian

In [ ]:
pool_ops, _ = ansatz.excitation_ops()
num_excited = min(len(pool_ops), 5)
selected_ops = pool_ops[:num_excited]

n_qubits = qubit_op.num_qubits
subspace_dim = len(selected_ops) + 1
subspace_hamiltonian = np.zeros((subspace_dim, subspace_dim))

print(f"Building {subspace_dim}x{subspace_dim} subspace Hamiltonian...")

for i in range(subspace_dim):
    for j in range(subspace_dim):
        if i == 0 and j == 0:
            subspace_hamiltonian[i, j] = result_vqe.eigenvalue.real
        elif i == 0 and j > 0:
            op_mat = selected_ops[j-1].to_matrix()
            subspace_hamiltonian[i, j] = float(np.trace(op_mat @ qubit_op.to_matrix())) / (2**n_qubits)
        elif i > 0 and j == 0:
            op_mat_dag = selected_ops[i-1].to_matrix().conj().T
            subspace_hamiltonian[i, j] = float(np.trace(op_mat_dag @ qubit_op.to_matrix())) / (2**n_qubits)
        elif i > 0 and j > 0:
            op_i = selected_ops[i-1].to_matrix()
            op_j_dag = selected_ops[j-1].to_matrix().conj().T
            subspace_hamiltonian[i, j] = float(np.trace(op_i @ op_j_dag @ qubit_op.to_matrix())) / (2**n_qubits)

print("Subspace Hamiltonian (diagonal elements):")
for i in range(subspace_dim):
    print(f"  H[{i},{i}] = {subspace_hamiltonian[i,i]:.8f}")

subspace_eigenvalues, _ = np.linalg.eigh(subspace_hamiltonian)
print(f"\nSubspace expanded excited state energies (electronic):")
for j, ev in enumerate(subspace_eigenvalues[:4]):
    label = "Ground State" if j == 0 else f"Excited {j}"
    print(f"  {j}: {ev + nuclear_repulsion:.10f} Ha")

## Step 7 — Energy Level Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

states = ["S₀", "S₁", "S₂", "S₃", "S₄"]
exact_plt = [all_energies[j] for j in range(5)]
subspace_plt = [subspace_eigenvalues[j] + nuclear_repulsion for j in range(5)]

x_exact = [1] * 5
x_subspace = [2] * 5

ax.scatter(x_exact, exact_plt, s=120, color="black", zorder=5, label="Exact (Classical)")
ax.scatter(x_subspace, subspace_plt, s=100, marker="x", color="blue", zorder=5,
           label="Subspace Expansion")

for j in range(5):
    ax.plot([1, 2], [exact_plt[j], subspace_plt[j]], "gray", linewidth=0.8, alpha=0.6)

ax.set_xticks([1, 2])
ax.set_xticklabels(["Exact Diagonalization", "Subspace Expansion"], fontsize=12)
ax.set_ylabel("Energy (Hartree)", fontsize=12)
ax.set_title("H₂ Excited State Energy Levels", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

for j, E in enumerate(exact_plt):
    ax.annotate(states[j], (1, E), textcoords="offset points", xytext=(-20, 5),
               fontsize=10)

plt.tight_layout()
plt.savefig("vqe_h2/excited_states_diagram.png", dpi=150)
plt.show()

print("Energy level diagram saved to excited_states_diagram.png")

## Step 8 — Excitation Energy Comparison

In [ ]:
print("=" * 70)
print(f"{'EXCITED STATE SUMMARY':^70}")
print("=" * 70)
print(f"{'State':<10} {'Exact (Ha)':<20} {'Subspace (Ha)':<20} {'Error (Ha)':<15}")
print("-" * 70)
for j in range(min(5, len(subspace_eigenvalues))):
    exact_exc = all_energies[j]
    sub_exc = subspace_eigenvalues[j] + nuclear_repulsion
    err = abs(exact_exc - sub_exc)
    print(f"S{j:<9} {exact_exc:<20.10f} {sub_exc:<20.10f} {err:<15.10f}")
print("=" * 70)